# Poisoned Chalice 2026 - Comprehensive MIA Suite
## All experiments with fixed ICP signal direction

**⚠️ IMPORTANT: Setup Instructions**

1. **Add Dataset** (REQUIRED):
   - Click "+ Add Data" button (top right)
   - Search: `AISE-TUDelft/Poisoned-Chalice`
   - Click "Add" to attach to notebook
   - ⚠️ Without this step, the notebook will fail!

2. **Enable GPU**:
   - Settings → Accelerator → GPU T4 x2

3. **Enable Internet**:
   - Turn ON (for model download)

---

**What this notebook does:**
- ✅ **Auto-detects dataset path** (handles multiple Kaggle folder structures)
- ✅ Tests 5 individual MIA methods (Loss, SURP, Min-K%++, ICP-Fixed, Gradient Norm)
- ✅ Creates rank ensemble with weighted combination
- ✅ Implements BUZZER-style multi-signal attack with calibration
- ✅ Automatically exports best method as submission

**Expected Runtime:** ~60-90 minutes (with 0.1 sample fraction)

---

**Troubleshooting:**
- If "Dataset not found" error: Make sure you added the dataset (step 1)
- If GPU out of memory: Reduce `SAMPLE_FRACTION` in cell 3
- If taking too long: Increase `SAMPLE_FRACTION` for faster testing

In [ ]:
# Environment setup
import os
import torch
import numpy as np
import pandas as pd
from datasets import load_from_disk
from transformers import AutoModelForCausalLM, AutoTokenizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import roc_auc_score
from scipy.stats import rankdata
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Configuration
MODEL_NAME = "bigcode/starcoder2-3b"
SAMPLE_FRACTION = 0.1
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MAX_LENGTH = 2048
BATCH_SIZE = 4

print(f"Device: {DEVICE}")
print(f"Sample Fraction: {SAMPLE_FRACTION}")

In [ ]:
# Load model and tokenizer
print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True
)
model.eval()

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    
print(f"Model loaded: {MODEL_NAME}")
print(f"Vocab size: {len(tokenizer)}")

In [ ]:
# Load dataset
print("Loading dataset...")

# Auto-detect dataset path
import os
from pathlib import Path

def find_python_dataset(root="/kaggle/input"):
    """Auto-detect Python dataset location"""
    # Try common paths first (fast)
    quick_paths = [
        "/kaggle/input/poisoned-chalice-dataset/poisoned_chalice_dataset/Python",
        "/kaggle/input/datasets/minh2duy/poisoned-chalice-dataset/poisoned_chalice_dataset/Python",
        "/kaggle/input/datasets/minh2duy/poisoned_chalice_dataset/Python",
        "/kaggle/input/aise-tudelft-poisoned-chalice/poisoned_chalice_dataset/Python",
    ]
    
    for path in quick_paths:
        if os.path.exists(path):
            print(f"✓ Found dataset (quick): {path}")
            return path
    
    # Deep search if quick paths fail
    print("⏳ Searching for dataset (this may take a moment)...")
    for dirpath, dirnames, filenames in os.walk(root):
        if 'Python' in dirnames:
            python_path = os.path.join(dirpath, 'Python')
            # Verify it's the correct dataset by checking for train/test splits
            try:
                python_contents = os.listdir(python_path)
                if any('train' in item.lower() for item in python_contents):
                    print(f"✓ Found dataset: {python_path}")
                    return python_path
            except:
                pass
    
    return None

dataset_path = find_python_dataset()

if dataset_path is None:
    # Show directory structure for debugging
    print("\n❌ Dataset not found! Directory structure:")
    if os.path.exists("/kaggle/input"):
        for item in os.listdir("/kaggle/input")[:5]:  # Show first 5 items
            item_path = os.path.join("/kaggle/input", item)
            print(f"  - {item}")
            if os.path.isdir(item_path):
                try:
                    for subitem in os.listdir(item_path)[:5]:
                        print(f"    -> {subitem}")
                except:
                    pass
    
    raise FileNotFoundError(
        "Please add the 'AISE-TUDelft/Poisoned-Chalice' dataset to this Kaggle notebook.\n"
        "Go to: Add Data → Search 'AISE-TUDelft/Poisoned-Chalice' → Add"
    )

# Load dataset
dataset = load_from_disk(dataset_path)

# Sample data
train_data = dataset['train']
if SAMPLE_FRACTION < 1.0:
    n_samples = int(len(train_data) * SAMPLE_FRACTION)
    indices = np.random.choice(len(train_data), n_samples, replace=False)
    train_data = train_data.select(indices)

test_data = dataset['test']
if SAMPLE_FRACTION < 1.0:
    n_samples_test = int(len(test_data) * SAMPLE_FRACTION)
    indices_test = np.random.choice(len(test_data), n_samples_test, replace=False)
    test_data = test_data.select(indices_test)

# Detect text field name (text vs content)
TEXT_FIELD = None
for candidate in ["text", "content"]:
    if candidate in train_data.column_names:
        TEXT_FIELD = candidate
        break
if TEXT_FIELD is None:
    # Fallback: first string-like field
    for col in train_data.column_names:
        if isinstance(train_data[0][col], str):
            TEXT_FIELD = col
            break

if TEXT_FIELD is None:
    raise KeyError(f"No text field found in dataset columns: {train_data.column_names}")

print(f"\n✅ Dataset loaded successfully!")
print(f"Train samples: {len(train_data)}")
print(f"Test samples: {len(test_data)}")
print(f"Text field: {TEXT_FIELD}")

def get_text(example):
    return example[TEXT_FIELD]

## Attack Implementations

In [ ]:
def compute_loss_score(text: str) -> float:
    """Basic loss-based MIA"""
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=MAX_LENGTH).to(DEVICE)
    with torch.no_grad():
        outputs = model(**inputs, labels=inputs["input_ids"])
        return -outputs.loss.item()

def compute_surp_score(text: str) -> float:
    """SURP: mean - std of log probs"""
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=MAX_LENGTH).to(DEVICE)
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        log_probs = torch.log_softmax(logits, dim=-1)
        target_log_probs = torch.gather(log_probs[:, :-1, :], 2, inputs["input_ids"][:, 1:].unsqueeze(-1)).squeeze(-1)
        mean_lp = target_log_probs.mean().item()
        std_lp = target_log_probs.std().item()
        return mean_lp - std_lp

In [ ]:
def compute_mink_plusplus_score(text: str, k: float = 0.2) -> float:
    """Min-K%++ with probability-weighted statistics (ICLR 2025)"""
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=MAX_LENGTH).to(DEVICE)
    
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits[:, :-1, :]  # [1, seq_len-1, vocab]
        probs = torch.softmax(logits, dim=-1)  # [1, seq_len-1, vocab]
        log_probs = torch.log_softmax(logits, dim=-1)  # [1, seq_len-1, vocab]
        
        # Probability-weighted mean and variance
        mu = (probs * log_probs).sum(dim=-1)  # [1, seq_len-1]
        var = (probs * (log_probs ** 2)).sum(dim=-1) - (mu ** 2)
        sigma = torch.sqrt(torch.clamp(var, min=1e-10))
        
        # Target token log probs
        target_ids = inputs["input_ids"][:, 1:].unsqueeze(-1)  # [1, seq_len-1, 1]
        target_log_probs = torch.gather(log_probs, 2, target_ids).squeeze(-1)  # [1, seq_len-1]
        
        # Z-score normalization
        z_scores = (target_log_probs - mu) / sigma  # [1, seq_len-1]
        
        # Select bottom k%
        k_length = max(1, int(z_scores.size(1) * k))
        mink_values, _ = torch.topk(z_scores, k_length, largest=False, sorted=False)
        
        return mink_values.mean().item()

In [ ]:
def compute_icp_score_fixed(text: str, num_probes: int = 5, mask_ratio: float = 0.3) -> float:
    """ICP-MIA-SP with FIXED score direction: probe_ll - base_ll
    
    Logic: Members reconstruct better from perturbed prefix
    -> Higher probe_ll for members -> MORE NEGATIVE gap
    -> Use probe_ll - base_ll so members have HIGHER scores
    """
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=MAX_LENGTH).to(DEVICE)
    input_ids = inputs["input_ids"]
    seq_len = input_ids.size(1)
    
    # Base log-likelihood
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits[:, :-1, :]
        log_probs = torch.log_softmax(logits, dim=-1)
        target_log_probs = torch.gather(log_probs, 2, input_ids[:, 1:].unsqueeze(-1)).squeeze(-1)
        base_ll = target_log_probs.mean().item()
    
    # Probe with random masking
    probe_lls = []
    for _ in range(num_probes):
        # Random mask in first half
        prefix_len = seq_len // 2
        num_mask = max(1, int(prefix_len * mask_ratio))
        mask_positions = np.random.choice(prefix_len, num_mask, replace=False)
        
        probe_ids = input_ids.clone()
        probe_ids[0, mask_positions] = tokenizer.mask_token_id if hasattr(tokenizer, 'mask_token_id') else tokenizer.unk_token_id
        
        with torch.no_grad():
            outputs = model(input_ids=probe_ids)
            logits = outputs.logits[:, :-1, :]
            log_probs = torch.log_softmax(logits, dim=-1)
            target_log_probs = torch.gather(log_probs, 2, input_ids[:, 1:].unsqueeze(-1)).squeeze(-1)
            probe_ll = target_log_probs.mean().item()
            probe_lls.append(probe_ll)
    
    # FIXED: Use probe_ll - base_ll (members have higher scores)
    min_probe_ll = min(probe_lls)
    return min_probe_ll - base_ll

In [ ]:
def compute_gradient_norm(text: str) -> float:
    """White-box gradient norm attack"""
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=MAX_LENGTH).to(DEVICE)
    
    # Get embeddings with gradients
    embeddings = model.get_input_embeddings()(inputs["input_ids"])
    embeddings.requires_grad = True
    
    # Forward pass
    outputs = model(inputs_embeds=embeddings, labels=inputs["input_ids"])
    loss = outputs.loss
    
    # Backward pass
    loss.backward()
    
    # Gradient norm
    grad_norm = embeddings.grad.norm(2).item()
    
    # Clean up
    model.zero_grad()
    
    return -grad_norm  # Negative because higher norm = member

## Experiment 1: Individual Baselines

In [ ]:
print("\n=== Running Individual Baselines ===")

# Compute scores for training data
train_scores = {
    'loss': [],
    'surp': [],
    'mink++': [],
    'icp_fixed': [],
    'grad_norm': []
}

print("\nProcessing train data...")
for example in tqdm(train_data, desc="Train"):
    text = get_text(example)
    train_scores['loss'].append(compute_loss_score(text))
    train_scores['surp'].append(compute_surp_score(text))
    train_scores['mink++'].append(compute_mink_plusplus_score(text))
    train_scores['icp_fixed'].append(compute_icp_score_fixed(text))
    train_scores['grad_norm'].append(compute_gradient_norm(text))

# Compute scores for test data  
test_scores = {
    'loss': [],
    'surp': [],
    'mink++': [],
    'icp_fixed': [],
    'grad_norm': []
}

print("\nProcessing test data...")
for example in tqdm(test_data, desc="Test"):
    text = get_text(example)
    test_scores['loss'].append(compute_loss_score(text))
    test_scores['surp'].append(compute_surp_score(text))
    test_scores['mink++'].append(compute_mink_plusplus_score(text))
    test_scores['icp_fixed'].append(compute_icp_score_fixed(text))
    test_scores['grad_norm'].append(compute_gradient_norm(text))

In [ ]:
# Evaluate individual methods
print("\n=== Individual Method Results ===")

all_scores = {}
for method in train_scores.keys():
    combined = np.concatenate([train_scores[method], test_scores[method]])
    labels = np.concatenate([np.ones(len(train_scores[method])), np.zeros(len(test_scores[method]))])
    auc = roc_auc_score(labels, combined)
    all_scores[method] = combined
    print(f"{method:12s}: AUC = {auc:.4f}")

## Experiment 2: Rank Ensemble

In [ ]:
print("\n=== Rank Ensemble ===")

# Convert to percentile ranks
def to_percentile(scores):
    return rankdata(scores, method='average') / len(scores)

rank_scores = {}
for method in all_scores.keys():
    rank_scores[method] = to_percentile(all_scores[method])

# Weighted combination (empirical weights from previous results)
weights = {
    'loss': 0.1,
    'surp': 0.2,
    'mink++': 0.15,
    'icp_fixed': 0.25,
    'grad_norm': 0.3
}

ensemble_score = np.zeros(len(rank_scores['loss']))
for method, weight in weights.items():
    ensemble_score += weight * rank_scores[method]

labels = np.concatenate([np.ones(len(train_data)), np.zeros(len(test_data))])
auc_ensemble = roc_auc_score(labels, ensemble_score)
print(f"Rank Ensemble AUC: {auc_ensemble:.4f}")

## Experiment 3: BUZZER-Style Multi-Signal + Calibration

In [ ]:
print("\n=== BUZZER-Style Attack ===")

def compute_all_signals(text: str) -> dict:
    """Extract all BUZZER signals"""
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=MAX_LENGTH).to(DEVICE)
    
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits[:, :-1, :]
        probs = torch.softmax(logits, dim=-1)
        log_probs = torch.log_softmax(logits, dim=-1)
        
        target_ids = inputs["input_ids"][:, 1:].unsqueeze(-1)
        target_log_probs = torch.gather(log_probs, 2, target_ids).squeeze(-1)
        target_probs = torch.gather(probs, 2, target_ids).squeeze(-1)
        
        # Signal 1: Mean log prob
        mean_lp = target_log_probs.mean().item()
        
        # Signal 2: SURP
        surp = mean_lp - target_log_probs.std().item()
        
        # Signal 3: Min-K%++ (simplified)
        mu = (probs * log_probs).sum(dim=-1)
        var = (probs * (log_probs ** 2)).sum(dim=-1) - (mu ** 2)
        sigma = torch.sqrt(torch.clamp(var, min=1e-10))
        z_scores = (target_log_probs - mu) / sigma
        k_length = max(1, int(z_scores.size(1) * 0.2))
        mink_plusplus = torch.topk(z_scores, k_length, largest=False)[0].mean().item()
        
        # Signal 4: Negative entropy
        entropy = -(target_probs * target_log_probs).sum(dim=-1).mean().item()
        neg_entropy = -entropy
        
    return {
        'mean_lp': mean_lp,
        'surp': surp,
        'mink_plusplus': mink_plusplus,
        'neg_entropy': neg_entropy
    }

def calibrate_signals(text: str, drop_ratio: float = 0.1) -> dict:
    """Compute signals with token dropping calibration"""
    original_signals = compute_all_signals(text)
    
    # Perturbed version
    tokens = tokenizer.encode(text)
    n_drop = max(1, int(len(tokens) * drop_ratio))
    drop_indices = np.random.choice(len(tokens), n_drop, replace=False)
    perturbed_tokens = [t for i, t in enumerate(tokens) if i not in drop_indices]
    perturbed_text = tokenizer.decode(perturbed_tokens, skip_special_tokens=True)
    
    perturbed_signals = compute_all_signals(perturbed_text)
    
    # Signal differences (calibration)
    calibrated = {}
    for key in original_signals:
        calibrated[f"{key}_orig"] = original_signals[key]
        calibrated[f"{key}_diff"] = original_signals[key] - perturbed_signals[key]
    
    return calibrated

In [ ]:
# Extract BUZZER features for training set (20% for LR training)
print("\nExtracting BUZZER features...")
n_train_lr = int(len(train_data) * 0.2)
train_lr_indices = np.random.choice(len(train_data), n_train_lr, replace=False)
train_lr_mask = np.zeros(len(train_data), dtype=bool)
train_lr_mask[train_lr_indices] = True

buzzer_features_train = []
buzzer_labels_train = []

for i, example in enumerate(tqdm(train_data, desc="Train BUZZER")):
    features = calibrate_signals(get_text(example))
    buzzer_features_train.append(list(features.values()))
    buzzer_labels_train.append(1)

buzzer_features_test = []
buzzer_labels_test = []

for example in tqdm(test_data, desc="Test BUZZER"):
    features = calibrate_signals(get_text(example))
    buzzer_features_test.append(list(features.values()))
    buzzer_labels_test.append(0)

X_train = np.array(buzzer_features_train)
y_train = np.array(buzzer_labels_train)
X_test = np.array(buzzer_features_test)
y_test = np.array(buzzer_labels_test)

In [ ]:
# Train Logistic Regression
print("\nTraining BUZZER classifier...")

# Use only subset for LR training
X_lr = X_train[train_lr_mask]
y_lr = y_train[train_lr_mask]

# Create stratified training samples
sss = StratifiedShuffleSplit(n_splits=3, test_size=0.2, random_state=42)
lr_models = []

for train_idx, val_idx in sss.split(X_lr, y_lr):
    lr = LogisticRegression(max_iter=1000, random_state=42)
    lr.fit(X_lr[train_idx], y_lr[train_idx])
    lr_models.append(lr)

# Predict with ensemble
X_all = np.vstack([X_train, X_test])
y_all = np.concatenate([y_train, y_test])

predictions = np.zeros(len(X_all))
for lr in lr_models:
    predictions += lr.predict_proba(X_all)[:, 1]
predictions /= len(lr_models)

auc_buzzer = roc_auc_score(y_all, predictions)
print(f"BUZZER AUC: {auc_buzzer:.4f}")

## Final Results Summary

In [ ]:
print("\n" + "="*60)
print("COMPREHENSIVE RESULTS SUMMARY")
print("="*60)

results_df = pd.DataFrame({
    'Method': ['Loss', 'SURP', 'Min-K%++', 'ICP-Fixed', 'Grad Norm', 'Rank Ensemble', 'BUZZER'],
    'AUC': [
        roc_auc_score(labels, all_scores['loss']),
        roc_auc_score(labels, all_scores['surp']),
        roc_auc_score(labels, all_scores['mink++']),
        roc_auc_score(labels, all_scores['icp_fixed']),
        roc_auc_score(labels, all_scores['grad_norm']),
        auc_ensemble,
        auc_buzzer
    ]
})

results_df = results_df.sort_values('AUC', ascending=False)
print(results_df.to_string(index=False))

# Save results
results_df.to_csv('comprehensive_results.csv', index=False)
print("\nResults saved to comprehensive_results.csv")

## Export Submission (Best Method)

In [ ]:
# Find best method
best_method = results_df.iloc[0]['Method']
print(f"\nBest method: {best_method}")

# Use appropriate scores for submission
if best_method == 'BUZZER':
    submission_scores = predictions
elif best_method == 'Rank Ensemble':
    submission_scores = ensemble_score
else:
    method_key = best_method.lower().replace('-', '_').replace(' ', '_')
    submission_scores = all_scores[method_key]

# Create submission
submission_df = pd.DataFrame({
    'id': list(range(len(train_data))) + list(range(len(test_data))),
    'score': submission_scores
})

submission_df.to_csv('submission.csv', index=False)
print(f"Submission saved with {best_method} scores")
print(f"Final AUC: {results_df.iloc[0]['AUC']:.4f}")

In [ ]:
# Sanity checks and final verification
print("\n" + "="*60)
print("SANITY CHECKS")
print("="*60)

# Check 1: Score distribution
print("\n1. Score Distribution Check:")
print(f"   Min score: {submission_scores.min():.4f}")
print(f"   Max score: {submission_scores.max():.4f}")
print(f"   Mean score: {submission_scores.mean():.4f}")
print(f"   Std score: {submission_scores.std():.4f}")

# Check 2: No NaN values
nan_count = np.isnan(submission_scores).sum()
print(f"\n2. NaN Check: {nan_count} NaNs (should be 0)")
if nan_count > 0:
    print("   ⚠️ Warning: NaN values detected! Filling with mean...")
    submission_scores = np.nan_to_num(submission_scores, nan=np.nanmean(submission_scores))

# Check 3: Submission file size
print(f"\n3. Submission size: {len(submission_df)} rows")
print(f"   Expected: {len(train_data) + len(test_data)} rows")
if len(submission_df) == len(train_data) + len(test_data):
    print("   ✅ Size matches!")
else:
    print("   ⚠️ Size mismatch!")

# Check 4: Performance vs baselines
print(f"\n4. Performance vs Random Baseline:")
print(f"   Best AUC: {results_df.iloc[0]['AUC']:.4f}")
if results_df.iloc[0]['AUC'] > 0.5:
    print(f"   ✅ Better than random (0.5000)")
else:
    print(f"   ⚠️ Worse than random - check implementation!")

print("\n" + "="*60)
print("✅ All checks complete! Ready to submit.")
print("="*60)

## 📊 Optional: Visualizations & Debugging

Run the cell below if you want to see score distributions and method comparisons.

In [ ]:
# Optional: Visualizations
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Plot 1: Method Comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart of AUCs
ax1 = axes[0]
results_df_sorted = results_df.sort_values('AUC', ascending=True)
colors = ['#ff6b6b' if auc < 0.6 else '#4ecdc4' if auc < 0.65 else '#95e1d3' for auc in results_df_sorted['AUC']]
ax1.barh(results_df_sorted['Method'], results_df_sorted['AUC'], color=colors, alpha=0.8)
ax1.axvline(x=0.5, color='red', linestyle='--', linewidth=2, label='Random Baseline')
ax1.set_xlabel('AUC Score', fontsize=12)
ax1.set_title('Method Performance Comparison', fontsize=14, fontweight='bold')
ax1.legend()
ax1.grid(axis='x', alpha=0.3)

# Score distribution for best method
ax2 = axes[1]
train_scores_best = submission_scores[:len(train_data)]
test_scores_best = submission_scores[len(train_data):]
ax2.hist(train_scores_best, bins=30, alpha=0.6, label='Members (train)', color='blue', edgecolor='black')
ax2.hist(test_scores_best, bins=30, alpha=0.6, label='Non-members (test)', color='red', edgecolor='black')
ax2.set_xlabel('Score', fontsize=12)
ax2.set_ylabel('Frequency', fontsize=12)
ax2.set_title(f'Score Distribution: {best_method}', fontsize=14, fontweight='bold')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('method_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✅ Visualization saved as 'method_comparison.png'")